In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import fontManager
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
from pathlib import Path

# Load Linux Libertine font
FONT_DIR = Path(__file__).parent / 'fonts' if '__file__' in dir() else Path('fonts')
for f in FONT_DIR.glob('*.otf'):
    fontManager.addfont(str(f))

FONT_NAME = 'Linux Libertine O'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set(
    context='paper',
    style='ticks',
    palette='deep',
    font=FONT_NAME,
    font_scale=1.8,
    rc={
        'mathtext.fontset': 'stix',
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.labelweight': 'normal',
    }
)
print(f'Font: {FONT_NAME}, Output: {OUTPUT_DIR}')

In [ ]:
# ---- 7-subplot line chart: BM25+Combined Hit@K across datasets ----
import matplotlib.ticker as mticker

df = pd.read_parquet('../../data/eval_results/exports/aggregated/all_experiments_combined.parquet')

# Filter: BM25 + combined only
df_plot = df[(df['retriever_type'] == 'bm25') & (df['hyde_mode'] == 'combined')].copy()

DATASET_LABELS = {
    'adventure_works': 'Ad.', 'bird': 'BD.', 'chembl': 'Ch.',
    'chicago': 'Cc.', 'fetaqa': 'FL.', 'fetaqapn': 'FM.',
    'public_bi': 'Pb.',
}
DATASET_ORDER = ['adventure_works', 'chembl', 'public_bi', 'fetaqa', 'fetaqapn', 'bird', 'chicago']

# 4 variants: 2 colors (Stage2 LLM) x 2 linestyles (Stage1 LLM)
COLOR_32B = '#C0392B'   # red — Qwen3-32B
COLOR_80B = '#3C6E8F'   # teal blue — Qwen3-Next-80B

VARIANTS = [
    ('gemini3flash_qwen332b',          COLOR_32B, '-',  'o'),
    ('gemini31flashlite_qwen332b',     COLOR_32B, '--', 's'),
    ('gemini3flash_qwen3next80b',      COLOR_80B, '-',  'o'),
    ('gemini31flashlite_qwen3next80b', COLOR_80B, '--', 's'),
]

VARIANT_LABELS = {
    'gemini3flash_qwen332b':          'Gemini3.0Flash + Qwen3-32B',
    'gemini31flashlite_qwen332b':     'Gemini3.1FlashLite + Qwen3-32B',
    'gemini3flash_qwen3next80b':      'Gemini3.0Flash + Qwen3-Next-80B',
    'gemini31flashlite_qwen3next80b': 'Gemini3.1FlashLite + Qwen3-Next-80B',
}

ks = [1, 5, 10]
k_cols = ['hit@1', 'hit@5', 'hit@10']

# Print data table
print(f"{'Dataset':<6} | {'Variant':<40} | {'Hit@1':>7} | {'Hit@5':>7} | {'Hit@10':>7}")
print('-' * 78)
for ds in DATASET_ORDER:
    for exp, _, _, _ in VARIANTS:
        row = df_plot[(df_plot['experiment'] == exp) & (df_plot['dataset'] == ds)]
        if row.empty:
            continue
        vals = [row[c].values[0] * 100 for c in k_cols]
        print(f"{DATASET_LABELS[ds]:<6} | {VARIANT_LABELS[exp]:<40} | {vals[0]:>7.2f} | {vals[1]:>7.2f} | {vals[2]:>7.2f}")

# Summary: best / worst / avg per dataset
print()
print(f"{'Dataset':<6} | {'Metric':<6} | {'Hit@1':>7} | {'Hit@5':>7} | {'Hit@10':>7} | {'Best Variant (by Hit@1)'}")
print('-' * 95)
for ds in DATASET_ORDER:
    all_vals = []
    for exp, _, _, _ in VARIANTS:
        row = df_plot[(df_plot['experiment'] == exp) & (df_plot['dataset'] == ds)]
        if row.empty:
            continue
        v = [row[c].values[0] * 100 for c in k_cols]
        all_vals.append((exp, v))
    if not all_vals:
        continue
    h1 = [v[0] for _, v in all_vals]
    h5 = [v[1] for _, v in all_vals]
    h10 = [v[2] for _, v in all_vals]
    best_idx = h1.index(max(h1))
    worst_idx = h1.index(min(h1))
    print(f"{DATASET_LABELS[ds]:<6} | {'Best':<6} | {max(h1):>7.2f} | {max(h5):>7.2f} | {max(h10):>7.2f} | {VARIANT_LABELS[all_vals[best_idx][0]]}")
    print(f"{'':6} | {'Worst':<6} | {min(h1):>7.2f} | {min(h5):>7.2f} | {min(h10):>7.2f} | {VARIANT_LABELS[all_vals[worst_idx][0]]}")
    print(f"{'':6} | {'Avg':<6} | {sum(h1)/len(h1):>7.2f} | {sum(h5)/len(h5):>7.2f} | {sum(h10)/len(h10):>7.2f} |")
print()

fig, axes = plt.subplots(1, 7, figsize=(7.09, 1.25))

for idx, ds in enumerate(DATASET_ORDER):
    ax = axes[idx]

    y_vals = []
    for exp, color, ls, marker in VARIANTS:
        row = df_plot[(df_plot['experiment'] == exp) & (df_plot['dataset'] == ds)]
        if row.empty:
            continue
        vals = [row[c].values[0] * 100 for c in k_cols]
        y_vals.extend(vals)
        ax.plot(ks, vals, color=color, linestyle=ls, marker=marker,
                markersize=3, linewidth=1.1, zorder=3)

    margin = (max(y_vals) - min(y_vals)) * 0.15
    ax.set_ylim(min(y_vals) - margin, max(y_vals) + margin)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=5))

    ax.set_xlabel(DATASET_LABELS[ds], fontsize=7, labelpad=1)
    ax.set_xticks(ks)
    ax.set_xticklabels(['1', '5', '10'], fontsize=6.5)
    ax.tick_params(axis='x', pad=1, direction='in')
    ax.tick_params(axis='y', labelsize=6.5, pad=0.5, direction='in')
    ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

axes[0].set_ylabel('Hit@K (%)', fontsize=6.5, labelpad=1)

# Compact legend: color = Qwen variant, linestyle = Gemini variant
legend_handles = [
    Line2D([0], [0], color=COLOR_32B, linestyle='-', marker='o', markersize=3.5,
           linewidth=1.1, label='Qwen3-32B'),
    Line2D([0], [0], color=COLOR_80B, linestyle='-', marker='o', markersize=3.5,
           linewidth=1.1, label='Qwen3-Next-80B'),
    Line2D([0], [0], color='#555555', linestyle='-',  marker='o', markersize=3.5,
           linewidth=1.1, label='Gemini 3.0 Flash'),
    Line2D([0], [0], color='#555555', linestyle='--', marker='s', markersize=3.5,
           linewidth=1.1, label='Gemini 3.1 Flash Lite'),
]
fig.legend(handles=legend_handles, loc='upper center', ncol=4,
           frameon=True, fontsize=6, handlelength=1.5, handletextpad=0.3,
           columnspacing=0.6,
           fancybox=False, edgecolor='#BBBBBB', borderpad=0.15,
           bbox_to_anchor=(0.5, 0.995))

fig.subplots_adjust(wspace=0.35, left=0.06, right=0.98, bottom=0.20, top=0.86)
fig.savefig(OUTPUT_DIR / 'retrieval_hitk_bm25_combined.pdf',
            bbox_inches='tight', pad_inches=0.01, dpi=300)
plt.show()
print(f'Saved to {OUTPUT_DIR / "retrieval_hitk_bm25_combined.pdf"}')